In [ ]:
# ==============================================================================
# FTSE 100 VOLATILITY FORECASTING
# Model: HAR-X (primary) + LSTM Residual Correction (secondary)
# Forecast: 3-Month Direct Multi-Step
# ==============================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import jarque_bera, shapiro, kstest, norm as scipy_norm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.stattools import durbin_watson
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

np.random.seed(42)
tf.random.set_seed(42)

# ==============================================================================
# CONFIGURATION
# ==============================================================================

DATA_PATH   = "global_indices_prices.csv"
TARGET_COL  = "^FTSE"
TARGET_NAME = "FTSE 100"
HORIZON     = 66        # 3 months = 66 trading days
SEQ_LEN_RES = 5         # residual LSTM lookback window
BATCH_SIZE  = 32
EPOCHS      = 200

LOG_FILE    = "log.txt"
EXCEL_FILE  = "clean_preprocessed_data.xlsx"
PLOTS_DIR   = "plots"

INDICES = {
    "^GSPC":     "S&P500",
    "^NDX":      "Nasdaq100",
    "^STOXX50E": "EuroStoxx50",
    "^FTSE":     "FTSE100",
    "^GDAXI":    "DAX",
    "^N225":     "Nikkei225",
    "000001.SS": "Shanghai",
    "^NSEI":     "Nifty50",
    "^HSI":      "HangSeng",
    "^AXJO":     "ASX200"
}

os.makedirs(PLOTS_DIR, exist_ok=True)

# ==============================================================================
# LOGGER SETUP
# Redirects all output exclusively to log.txt. No console output.
# ==============================================================================

log = open(LOG_FILE, "w", buffering=1, encoding="utf-8")

def wlog(msg=""):
    log.write(str(msg) + "\n")

def wlog_sep(title=""):
    log.write("\n" + "=" * 70 + "\n")
    if title:
        log.write(title + "\n")
        log.write("=" * 70 + "\n")

# Suppress all stdout/stderr from libraries (keras verbose etc.)
# We will pass verbose=0 to all model calls

# ==============================================================================
# SECTION 1: LOAD RAW DATA
# ==============================================================================

df_raw = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw = df_raw.sort_index()

AVAIL_COLS  = [c for c in INDICES.keys() if c in df_raw.columns]
AVAIL_NAMES = [INDICES[c] for c in AVAIL_COLS]

wlog_sep("1. RAW DATA SUMMARY")
wlog(f"Shape            : {df_raw.shape}")
wlog(f"Date range       : {df_raw.index[0].date()} to {df_raw.index[-1].date()}")
wlog(f"Total trading days: {len(df_raw)}")
wlog(f"Available indices: {AVAIL_COLS}")
wlog(f"\nMissing values per column (raw):")
for col in AVAIL_COLS:
    pct = df_raw[col].isnull().mean() * 100
    wlog(f"  {INDICES[col]:>15s} ({col}): {df_raw[col].isnull().sum():>5d}  ({pct:.2f}%)")


# ==============================================================================
# SECTION 2: DATA CLEANING — STEP 1: STRUCTURAL ISSUES
# ==============================================================================
# Fix index, duplicates, non-positive prices before any analysis.

df = df_raw[AVAIL_COLS].copy()

# Remove duplicate dates, keep first occurrence
n_before = len(df)
df = df[~df.index.duplicated(keep="first")]
n_dup_removed = n_before - len(df)

# Non-positive prices are physically impossible, set to NaN
zero_mask   = df <= 0
zero_counts = zero_mask.sum()
df[zero_mask] = np.nan

wlog_sep("2. CLEANING — STRUCTURAL")
wlog(f"Duplicate dates removed : {n_dup_removed}")
wlog(f"Non-positive prices → NaN:")
for col in AVAIL_COLS:
    wlog(f"  {INDICES[col]:>15s}: {zero_counts[col]}")


# ==============================================================================
# SECTION 3: DATA CLEANING — STEP 2: SPIKE DETECTION
# ==============================================================================
# Single-day log-return > 50% in absolute value is treated as a data error.
# Market circuits breakers prevent real moves of this magnitude.
# Set these cells to NaN for interpolation.

spike_counts = {}
for col in AVAIL_COLS:
    lr         = np.log(df[col] / df[col].shift(1))
    spike_mask = lr.abs() > 0.50
    spike_counts[col] = int(spike_mask.sum())
    df.loc[spike_mask, col] = np.nan

wlog_sep("3. CLEANING — SPIKE DETECTION (|log-return| > 50%)")
for col in AVAIL_COLS:
    wlog(f"  {INDICES[col]:>15s}: {spike_counts[col]} spikes removed")


# ==============================================================================
# SECTION 4: DATA CLEANING — STEP 3: ROLLING Z-SCORE OUTLIER DETECTION
# ==============================================================================
# Compute 252-day rolling mean and std for each series.
# Any price that deviates > 5 rolling std is flagged as an outlier.
# This catches stale prices that persisted across missing data windows.
# Threshold of 5 sigma is conservative to avoid removing genuine vol spikes.

zscore_counts = {}
for col in AVAIL_COLS:
    roll_mean = df[col].rolling(252, min_periods=60).mean()
    roll_std  = df[col].rolling(252, min_periods=60).std()
    z         = (df[col] - roll_mean) / (roll_std + 1e-9)
    outlier   = z.abs() > 5
    zscore_counts[col] = int(outlier.sum())
    df.loc[outlier, col] = np.nan

wlog_sep("4. CLEANING — ROLLING Z-SCORE OUTLIERS (|z| > 5, window=252)")
for col in AVAIL_COLS:
    wlog(f"  {INDICES[col]:>15s}: {zscore_counts[col]} outliers removed")
total_removed = sum(zscore_counts.values())
wlog(f"  Total removed: {total_removed}")


# ==============================================================================
# SECTION 5: DATA CLEANING — STEP 4: IMPUTATION
# ==============================================================================
# Order: forward-fill (max 3 days) → backward-fill (max 3 days) → linear interp.
# Forward-fill handles missing trading days (holidays, exchange closures).
# Backward-fill and interpolation handle any remaining edge gaps.

df = df.ffill(limit=3).bfill(limit=3)
df = df.interpolate(method="linear", limit_direction="both", axis=0)

missing_after = df.isnull().sum().sum()

wlog_sep("5. CLEANING — IMPUTATION (ffill 3 → bfill 3 → linear interp)")
wlog(f"Missing values remaining after imputation: {missing_after}")
wlog(f"Clean shape: {df.shape}")


# ==============================================================================
# SECTION 6: OUTLIER WINSORIZATION ON LOG-RETURNS
# ==============================================================================
# Winsorize log-returns at 1st / 99th percentile to cap extreme observations.
# Applied to returns only (not price levels) to preserve series continuity.
# This is separate from spike removal: spikes were hard errors, this handles
# legitimate but extreme market moves that can distort rolling vol estimates.

lr_clean  = np.log(df / df.shift(1)).dropna()
lr_winsor = lr_clean.copy()
winsor_counts = {}
winsor_bounds = {}

for col in AVAIL_COLS:
    lo  = lr_clean[col].quantile(0.01)
    hi  = lr_clean[col].quantile(0.99)
    cnt = int(((lr_clean[col] < lo) | (lr_clean[col] > hi)).sum())
    lr_winsor[col]   = lr_clean[col].clip(lower=lo, upper=hi)
    winsor_counts[col] = cnt
    winsor_bounds[col] = (lo, hi)

wlog_sep("6. WINSORIZATION (log-returns, 1st/99th percentile)")
wlog(f"{'Index':>15}  {'Clipped':>8}  {'Lower bound':>13}  {'Upper bound':>13}")
wlog("-" * 56)
for col in AVAIL_COLS:
    lo, hi = winsor_bounds[col]
    wlog(f"  {INDICES[col]:>13}  {winsor_counts[col]:>8}  {lo:>13.6f}  {hi:>13.6f}")


# ==============================================================================
# SECTION 7: DESCRIPTIVE STATISTICS — PRICES
# ==============================================================================

price_stats = df.describe().T
price_stats.index = [INDICES[c] for c in AVAIL_COLS]
price_stats["skewness"] = df.skew()
price_stats["kurtosis"] = df.kurt()

wlog_sep("7. DESCRIPTIVE STATISTICS — PRICES")
wlog(price_stats[["mean", "std", "min", "25%", "50%", "75%",
                   "max", "skewness", "kurtosis"]].round(2).to_string())


# ==============================================================================
# SECTION 8: DESCRIPTIVE STATISTICS — LOG-RETURNS
# ==============================================================================

lr_stats = lr_clean.describe().T
lr_stats.index = [INDICES[c] for c in AVAIL_COLS]
lr_stats["skewness"]  = lr_clean.skew()
lr_stats["kurtosis"]  = lr_clean.kurt()
lr_stats["JB_stat"]   = [jarque_bera(lr_clean[c].dropna())[0] for c in AVAIL_COLS]
lr_stats["JB_pval"]   = [jarque_bera(lr_clean[c].dropna())[1] for c in AVAIL_COLS]
lr_stats["ann_vol"]   = lr_clean.std() * np.sqrt(252) * 100
lr_stats["ann_return"]= lr_clean.mean() * 252 * 100

wlog_sep("8. DESCRIPTIVE STATISTICS — LOG-RETURNS")
wlog(lr_stats[["mean", "std", "min", "max", "skewness", "kurtosis",
               "ann_return", "ann_vol", "JB_pval"]].round(4).to_string())

wlog("\nNormality interpretation: JB p-val < 0.05 = reject normality (heavy tails)")
for col in AVAIL_COLS:
    pval = lr_stats.loc[INDICES[col], "JB_pval"]
    interp = "Non-normal (fat tails)" if pval < 0.05 else "Cannot reject normality"
    wlog(f"  {INDICES[col]:>15s}: p={pval:.4f}  -> {interp}")


# ==============================================================================
# SECTION 9: STATIONARITY TESTS
# ==============================================================================
# ADF: null = unit root (non-stationary). p < 0.05 → stationary.
# KPSS: null = stationary. p < 0.05 → non-stationary.
# Both tests applied to log-returns.

wlog_sep("9. STATIONARITY TESTS (log-returns)")
wlog(f"{'Index':>15}  {'ADF stat':>10}  {'ADF p':>8}  {'KPSS stat':>10}  "
     f"{'KPSS p':>8}  {'Verdict':>14}")
wlog("-" * 75)

for col in AVAIL_COLS:
    series    = lr_clean[col].dropna()
    adf_s, adf_p, *_ = adfuller(series, autolag="AIC")
    kpss_s, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    # Both agree on stationary: ADF rejects H0(non-stat) and KPSS fails to reject H0(stat)
    adf_ok  = adf_p < 0.05
    kpss_ok = kpss_p > 0.05
    verdict = "Stationary" if (adf_ok and kpss_ok) else (
              "Likely stationary" if (adf_ok or kpss_ok) else "Non-stationary")
    wlog(f"  {INDICES[col]:>13}  {adf_s:>10.4f}  {adf_p:>8.4f}  "
         f"{kpss_s:>10.4f}  {kpss_p:>8.4f}  {verdict:>14}")


# ==============================================================================
# SECTION 10: ROLLING VOLATILITY STATISTICS
# ==============================================================================
# Compute 5-day, 22-day, 66-day annualized realized vol for all indices.
# Report current level, mean, std, and current percentile rank.

wlog_sep("10. REALIZED VOLATILITY STATISTICS (5-day, annualized)")
wlog(f"{'Index':>15}  {'Current%':>10}  {'Mean%':>8}  {'Std%':>8}  "
     f"{'Min%':>8}  {'Max%':>8}  {'Pct-rank':>10}")
wlog("-" * 75)

rv5_all = {}
for col in AVAIL_COLS:
    lr  = lr_clean[col]
    rv5 = lr.rolling(5).std() * np.sqrt(252) * 100
    rv5_all[col] = rv5
    curr = rv5.iloc[-1]
    pct  = (rv5.dropna() <= curr).mean() * 100
    wlog(f"  {INDICES[col]:>13}  {curr:>10.2f}  {rv5.mean():>8.2f}  "
         f"{rv5.std():>8.2f}  {rv5.min():>8.2f}  {rv5.max():>8.2f}  {pct:>8.1f}th")

wlog("\nCurrent FTSE volatility regime:")
ftse_rv5  = rv5_all[TARGET_COL].dropna()
curr_ftse = ftse_rv5.iloc[-1]
p25       = ftse_rv5.quantile(0.25)
p75       = ftse_rv5.quantile(0.75)
if curr_ftse < p25:
    regime = "LOW (below 25th pct)"
elif curr_ftse > p75:
    regime = "HIGH (above 75th pct)"
else:
    regime = "NORMAL (25th-75th pct)"
wlog(f"  Current: {curr_ftse:.2f}%  P25: {p25:.2f}%  P75: {p75:.2f}%  Regime: {regime}")


# ==============================================================================
# SECTION 11: VOLATILITY CORRELATION ANALYSIS
# ==============================================================================
# Cross-index correlation of 5-day realized vol.
# High correlation indicates strong volatility spillover channels.

rv5_df = pd.DataFrame(
    {INDICES[c]: rv5_all[c] for c in AVAIL_COLS}
).dropna()

corr_mat = rv5_df.corr()

wlog_sep("11. VOLATILITY CORRELATION MATRIX (5-day RV)")
wlog(corr_mat.round(3).to_string())

# FTSE pairwise correlations sorted
ftse_corrs = corr_mat["FTSE100"].drop("FTSE100").sort_values(ascending=False)
wlog(f"\nFTSE100 pairwise vol correlations (sorted):")
for name, val in ftse_corrs.items():
    wlog(f"  {name:>15s}: {val:.4f}")


# ==============================================================================
# SECTION 12: AUTOCORRELATION OF FTSE REALIZED VOL (LONG-MEMORY TEST)
# ==============================================================================
# High and slowly decaying ACF in realized vol confirms long memory,
# which is the theoretical basis for the HAR model.

ftse_rv5_log = np.log(rv5_all[TARGET_COL].dropna() / 100 + 1e-8)
acf_vals  = acf(ftse_rv5_log,  nlags=66, fft=True)
pacf_vals = pacf(ftse_rv5_log, nlags=40)

wlog_sep("12. FTSE LOG-VOL AUTOCORRELATION (long-memory evidence)")
wlog(f"ACF values at key lags:")
wlog(f"  Lag 1  : {acf_vals[1]:.4f}")
wlog(f"  Lag 5  : {acf_vals[5]:.4f}")
wlog(f"  Lag 22 : {acf_vals[22]:.4f}")
wlog(f"  Lag 66 : {acf_vals[66]:.4f}")
wlog(f"\nSlowly decaying ACF (still significant at lag 66 > 0.05) confirms")
wlog(f"long-memory in realized vol, justifying HAR model structure.")


# ==============================================================================
# SECTION 13: SAVE CLEAN DATA TO EXCEL
# ==============================================================================

with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:
    df.to_excel(writer,             sheet_name="Clean_Prices")
    lr_clean.to_excel(writer,       sheet_name="Log_Returns")
    price_stats.round(4).to_excel(writer, sheet_name="Price_Stats")
    lr_stats.round(6).to_excel(writer,    sheet_name="Return_Stats")
    corr_mat.round(4).to_excel(writer,    sheet_name="Vol_Correlation")

wlog_sep("13. FILES SAVED")
wlog(f"Excel: {EXCEL_FILE}  (5 sheets: Clean_Prices, Log_Returns, Price_Stats, Return_Stats, Vol_Correlation)")


# ==============================================================================
# SECTION 14: FEATURE ENGINEERING (10 FEATURES)
# ==============================================================================
# Features are built on clean price data (df), not on winsorized returns,
# because rolling std already smooths out extremes.
#
# FTSE HAR components (5):
#   ftse_log_rv5   : weekly realized vol — captures short-term vol clustering
#   ftse_log_rv22  : monthly realized vol — captures medium-term persistence
#   ftse_log_rv66  : quarterly realized vol — captures long-term mean reversion
#   ftse_neg_vol   : asymmetric leverage effect (negative-return days only)
#   ftse_vol_ratio : log(rv5/rv22) — regime signal: rising = short spike vs trend
#
# Cross-market (4):
#   spx_log_rv5    : S&P 500 weekly vol — strongest FTSE co-movement
#   dax_log_rv5    : DAX weekly vol — European market spillover
#   stoxx_log_rv5  : EuroStoxx50 — broad European vol signal
#   nikkei_log_rv5 : Nikkei weekly vol — overnight Asia-Pacific spillover
#
# Momentum (1):
#   ftse_mom5      : 5-day price momentum — trend-reversal signal for vol spikes

feat = pd.DataFrame(index=df.index)

p_ftse  = df[TARGET_COL]
lr_ftse = np.log(p_ftse / p_ftse.shift(1))
rv5_f   = lr_ftse.rolling(5).std()  * np.sqrt(252)
rv22_f  = lr_ftse.rolling(22).std() * np.sqrt(252)
rv66_f  = lr_ftse.rolling(66).std() * np.sqrt(252)

feat["ftse_log_rv5"]   = np.log(rv5_f  + 1e-8)
feat["ftse_log_rv22"]  = np.log(rv22_f + 1e-8)
feat["ftse_log_rv66"]  = np.log(rv66_f + 1e-8)
feat["ftse_neg_vol"]   = (lr_ftse.clip(upper=0)).abs() * np.sqrt(252)
feat["ftse_vol_ratio"] = np.log(rv5_f + 1e-8) - np.log(rv22_f + 1e-8)
feat["ftse_mom5"]      = p_ftse.pct_change(5)

for col, name in [("^GSPC", "spx"), ("^GDAXI", "dax"),
                  ("^STOXX50E", "stoxx"), ("^N225", "nikkei")]:
    if col in df.columns:
        lr_x = np.log(df[col] / df[col].shift(1))
        feat[f"{name}_log_rv5"] = np.log(lr_x.rolling(5).std() * np.sqrt(252) + 1e-8)

feat.dropna(inplace=True)

feature_cols  = list(feat.columns)
n_features    = len(feature_cols)
target_series = feat["ftse_log_rv5"].values
n_total       = len(feat)

current_vol = np.exp(feat["ftse_log_rv5"].iloc[-1]) * 100
hist_mean   = np.exp(feat["ftse_log_rv5"].mean()) * 100
hist_std_v  = feat["ftse_log_rv5"].std()

wlog_sep("14. FEATURE MATRIX")
wlog(f"Features ({n_features}): {feature_cols}")
wlog(f"Shape              : {feat.shape}")
wlog(f"Current FTSE vol   : {current_vol:.2f}%")
wlog(f"Historical mean    : {hist_mean:.2f}%")
wlog(f"Log-vol std (train): {hist_std_v:.4f}")
wlog(f"\nFeature stats (mean / std):")
for col in feature_cols:
    wlog(f"  {col:25s}: mean={feat[col].mean():.4f}  std={feat[col].std():.4f}")
wlog(f"\nFeature correlation with target (ftse_log_rv5):")
for col in feature_cols:
    c = feat[col].corr(feat["ftse_log_rv5"])
    wlog(f"  {col:25s}: {c:+.4f}")

feat.to_csv("feature_matrix.csv")


# ==============================================================================
# SECTION 15: TABULAR SEQUENCES FOR HAR-X
# ==============================================================================
# X_tab[t] = 10 features at day t               shape: (N-66, 10)
# y_tab[t] = [log_rv5(t+1), ..., log_rv5(t+66)] shape: (N-66, 66)
#
# No overlapping windows — each row is fully independent.
# Chronological 70/15/15 split preserves time ordering.

X_tab = feat[feature_cols].values[:-HORIZON]
y_tab = np.array([
    target_series[t + 1 : t + 1 + HORIZON]
    for t in range(len(X_tab))
])

n_tab   = len(X_tab)
n_train = int(n_tab * 0.70)
n_val   = int(n_tab * 0.15)
n_test  = n_tab - n_train - n_val

X_tr = X_tab[:n_train];                 y_tr = y_tab[:n_train]
X_va = X_tab[n_train:n_train+n_val];   y_va = y_tab[n_train:n_train+n_val]
X_te = X_tab[n_train+n_val:];          y_te = y_tab[n_train+n_val:]

scaler_tab = StandardScaler()
X_tr_sc    = scaler_tab.fit_transform(X_tr)
X_va_sc    = scaler_tab.transform(X_va)
X_te_sc    = scaler_tab.transform(X_te)

har_test_dates = feat.index[n_train + n_val : n_train + n_val + n_test]

train_start = feat.index[0].date()
train_end   = feat.index[n_train - 1].date()
val_end     = feat.index[n_train + n_val - 1].date()
test_end    = har_test_dates[-1]

wlog_sep("15. TRAIN / VAL / TEST SPLIT")
wlog(f"Train (70%) : {n_train} rows  [{train_start} to {train_end}]")
wlog(f"Val   (15%) : {n_val}  rows  [{train_end} to {val_end}]")
wlog(f"Test  (15%) : {n_test} rows  [{har_test_dates[0].date()} to {test_end}]")


# ==============================================================================
# SECTION 16: HAR-X DIRECT MULTI-STEP (PRIMARY MODEL)
# ==============================================================================
# Fits one Ridge regression per horizon h=1..66 via MultiOutputRegressor.
# Input at time t: 10 features (current day's known values).
# Output at time t: 66 future log-vol values (one per trading day).
# RidgeCV selects optimal L2 penalty via 5-fold cross-validation.
# No overlapping sequences → no information leakage → robust out-of-sample.

har = MultiOutputRegressor(
    RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5),
    n_jobs=-1
)
har.fit(X_tr_sc, y_tr)

y_har_tr  = har.predict(X_tr_sc)
y_har_va  = har.predict(X_va_sc)
y_har_te  = har.predict(X_te_sc)

# Selected alpha per horizon (first 3 and last as sample)
selected_alphas = [est.alpha_ for est in har.estimators_]

r2_har_h1  = r2_score(y_te[:, 0],  y_har_te[:, 0])
r2_har_h22 = r2_score(y_te[:, 21], y_har_te[:, 21])
r2_har_h66 = r2_score(y_te[:, 65], y_har_te[:, 65])
rmse_har   = np.sqrt(mean_squared_error(y_te[:, 0], y_har_te[:, 0]))
rmse_pct   = np.sqrt(mean_squared_error(
    np.exp(y_te[:, 0])*100, np.exp(y_har_te[:, 0])*100))
da_har     = np.mean(
    np.sign(np.diff(y_te[:, 0])) == np.sign(np.diff(y_har_te[:, 0]))) * 100
mae_har    = mean_absolute_error(y_te[:, 0], y_har_te[:, 0])

# Ridge coefficients at h=1 as feature importance proxy
ridge_h1    = har.estimators_[0]
coef_abs    = pd.Series(np.abs(ridge_h1.coef_), index=feature_cols).sort_values(ascending=False)
coef_signed = pd.Series(ridge_h1.coef_,         index=feature_cols)

# Durbin-Watson on HAR residuals at h=1 (checks for serial correlation in errors)
dw_stat = durbin_watson(y_te[:, 0] - y_har_te[:, 0])

wlog_sep("16. HAR-X MODEL RESULTS")
wlog(f"  R2  (h=1)   : {r2_har_h1:.4f}")
wlog(f"  R2  (h=22)  : {r2_har_h22:.4f}")
wlog(f"  R2  (h=66)  : {r2_har_h66:.4f}")
wlog(f"  RMSE (log)  : {rmse_har:.4f}")
wlog(f"  RMSE (vol%) : {rmse_pct:.3f}%")
wlog(f"  MAE  (log)  : {mae_har:.4f}")
wlog(f"  DA          : {da_har:.2f}%")
wlog(f"  Durbin-Watson (h=1 residuals): {dw_stat:.4f}  "
     f"(2.0=no autocorr, <1.5 or >2.5 = concern)")
wlog(f"\n  Selected RidgeCV alphas (h=1,5,22,66): "
     f"{selected_alphas[0]:.3f}, {selected_alphas[4]:.3f}, "
     f"{selected_alphas[21]:.3f}, {selected_alphas[65]:.3f}")
wlog(f"\nHAR-X Feature Importance (|coef| at h=1, sorted):")
for fname, val in coef_abs.items():
    sign = "+" if coef_signed[fname] >= 0 else "-"
    wlog(f"  {fname:25s}: {sign}{val:.4f}")


# ==============================================================================
# SECTION 17: HAR-X RESIDUALS FOR LSTM
# ==============================================================================
# LSTM trains exclusively on the residuals from HAR-X predictions.
# Residuals represent the nonlinear, short-memory patterns the linear
# HAR model fails to capture (regime switches, clustered extremes).
# Training on residuals (lower variance signal) prevents overfitting.

X_all_sc  = scaler_tab.transform(X_tab)
y_har_all = har.predict(X_all_sc)
residuals = y_tab - y_har_all     # shape: (N_tab, 66)

resid_var_ratio = residuals.std()**2 / np.var(target_series)
har_var_explained = (1 - resid_var_ratio) * 100

wlog_sep("17. HAR-X RESIDUALS SUMMARY")
wlog(f"Residual mean            : {residuals.mean():.6f}  (near 0 = unbiased)")
wlog(f"Residual std (log-vol)   : {residuals.std():.4f}")
wlog(f"Raw log-vol std          : {np.std(target_series):.4f}")
wlog(f"Variance explained by HAR: {har_var_explained:.1f}%")
wlog(f"Remaining for LSTM       : {100 - har_var_explained:.1f}%")

scaler_r = StandardScaler()
resid_sc = scaler_r.fit_transform(residuals)


# ==============================================================================
# SECTION 18: RESIDUAL LSTM SEQUENCES (stride=5)
# ==============================================================================
# Each training sample: X = last SEQ_LEN_RES=5 residual vectors (shape 5×66)
#                        y = next residual vector (shape 66)
# stride=5 samples every 5th window, reducing temporal correlation
# between adjacent training samples (week-level independence).

def make_resid_sequences(resid_sc, seq_len, stride):
    Xs, ys, idxs = [], [], []
    for i in range(seq_len, len(resid_sc), stride):
        Xs.append(resid_sc[i - seq_len : i])
        ys.append(resid_sc[i])
        idxs.append(i)
    return np.array(Xs), np.array(ys), np.array(idxs)

X_r, y_r, r_idxs = make_resid_sequences(resid_sc, SEQ_LEN_RES, stride=5)
n_r    = len(X_r)
n_r_tr = int(n_r * 0.70)
n_r_va = int(n_r * 0.15)

X_r_tr = X_r[:n_r_tr];               y_r_tr = y_r[:n_r_tr]
X_r_va = X_r[n_r_tr:n_r_tr+n_r_va]; y_r_va = y_r[n_r_tr:n_r_tr+n_r_va]
X_r_te = X_r[n_r_tr+n_r_va:];       y_r_te = y_r[n_r_tr+n_r_va:]
r_te_idxs = r_idxs[n_r_tr + n_r_va:]

wlog_sep("18. RESIDUAL LSTM SEQUENCES")
wlog(f"Total sequences: {n_r}  (stride=5, SEQ_LEN={SEQ_LEN_RES})")
wlog(f"Input shape    : ({SEQ_LEN_RES}, {HORIZON})")
wlog(f"Train          : {len(X_r_tr)}")
wlog(f"Val            : {len(X_r_va)}")
wlog(f"Test           : {len(X_r_te)}")


# ==============================================================================
# SECTION 19: BUILD AND TRAIN RESIDUAL LSTM
# ==============================================================================
# Architecture: LSTM(32) → BatchNorm → Dropout(0.3) → Dense(32) → Dense(66)
# LSTM(32): small enough to generalize on ~400 training sequences
# L2=1e-3 on both LSTM and Dense: prevents weight explosion on small dataset
# Huber loss: robust to the occasional large residual at spike horizons
# ModelCheckpoint: always restores best val_loss weights after full training

def build_resid_model(seq_len, n_out):
    inp = Input(shape=(seq_len, n_out))
    x   = LSTM(32, kernel_regularizer=regularizers.l2(1e-3))(inp)
    x   = BatchNormalization()(x)
    x   = Dropout(0.3)(x)
    x   = Dense(32, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-3))(x)
    out = Dense(n_out)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(0.001), loss="huber")
    return model

resid_model = build_resid_model(SEQ_LEN_RES, HORIZON)

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10,
                      min_lr=1e-6, verbose=0),
    ModelCheckpoint("best_resid_model.keras", monitor="val_loss",
                    save_best_only=True, verbose=0)
]

history = resid_model.fit(
    X_r_tr, y_r_tr,
    validation_data=(X_r_va, y_r_va),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=0
)

resid_model.load_weights("best_resid_model.keras")
best_epoch  = int(np.argmin(history.history["val_loss"])) + 1
best_val    = min(history.history["val_loss"])
final_train = history.history["loss"][-1]
train_losses = history.history["loss"]
val_losses   = history.history["val_loss"]

wlog_sep("19. RESIDUAL LSTM TRAINING")
wlog(f"Epochs run       : {EPOCHS}")
wlog(f"Best epoch       : {best_epoch}  (weights restored)")
wlog(f"Best val_loss    : {best_val:.4f}")
wlog(f"Final train_loss : {final_train:.4f}")
wlog(f"Train/val gap    : {final_train - best_val:.4f}  (negative = good generalization)")
wlog(f"\nLoss at every 10th epoch:")
for ep in range(9, EPOCHS, 10):
    if ep < len(train_losses):
        wlog(f"  Epoch {ep+1:>4d}: train={train_losses[ep]:.4f}  val={val_losses[ep]:.4f}")


# ==============================================================================
# SECTION 20: ENSEMBLE EVALUATION ON TEST SET
# ==============================================================================
# r_te_idxs maps each LSTM test sample to its row in the tabular arrays,
# ensuring HAR-X and LSTM predictions are computed for the same dates.
# Ensemble = HAR_prediction + LSTM_residual_correction

y_hat_r_sc = resid_model.predict(X_r_te, verbose=0)
y_hat_r    = scaler_r.inverse_transform(y_hat_r_sc)

valid_mask   = r_te_idxs < len(y_tab)
r_te_v       = r_te_idxs[valid_mask]
y_hat_r_v    = y_hat_r[valid_mask]

y_har_te_aln = y_har_all[r_te_v]
y_true_aln   = y_tab[r_te_v]
y_ens_te     = y_har_te_aln + y_hat_r_v
ens_test_dates = feat.index[r_te_v]

# Per-horizon R2 for ensemble
r2_ens_by_h = [r2_score(y_true_aln[:, h], y_ens_te[:, h]) for h in range(HORIZON)]

r2_ens_h1  = r2_ens_by_h[0]
r2_ens_h22 = r2_ens_by_h[21]
r2_ens_h66 = r2_ens_by_h[65]
rmse_ens   = np.sqrt(mean_squared_error(y_true_aln[:, 0], y_ens_te[:, 0]))
rmse_pct_e = np.sqrt(mean_squared_error(
    np.exp(y_true_aln[:, 0])*100, np.exp(y_ens_te[:, 0])*100))
da_ens     = np.mean(
    np.sign(np.diff(y_true_aln[:, 0])) == np.sign(np.diff(y_ens_te[:, 0]))) * 100
mae_ens    = mean_absolute_error(y_true_aln[:, 0], y_ens_te[:, 0])
mape_ens   = np.mean(np.abs((y_true_aln[:, 0] - y_ens_te[:, 0]) /
                             (np.abs(y_true_aln[:, 0]) + 1e-9))) * 100
dw_ens     = durbin_watson(y_true_aln[:, 0] - y_ens_te[:, 0])

# Improvement of ensemble over HAR-X alone
r2_improvement  = r2_ens_h1  - r2_har_h1
rmse_improvement = rmse_har  - rmse_ens

wlog_sep("20. ENSEMBLE RESULTS (TEST SET)")
wlog(f"  R2  (h=1)            : {r2_ens_h1:.4f}  (HAR-X alone: {r2_har_h1:.4f}  improvement: +{r2_improvement:.4f})")
wlog(f"  R2  (h=22)           : {r2_ens_h22:.4f}")
wlog(f"  R2  (h=66)           : {r2_ens_h66:.4f}")
wlog(f"  RMSE (log, h=1)      : {rmse_ens:.4f}  (HAR-X: {rmse_har:.4f}  improvement: {rmse_improvement:.4f})")
wlog(f"  RMSE (vol%, h=1)     : {rmse_pct_e:.3f}%")
wlog(f"  MAE  (log, h=1)      : {mae_ens:.4f}")
wlog(f"  MAPE (h=1)           : {mape_ens:.2f}%")
wlog(f"  DA   (h=1)           : {da_ens:.2f}%")
wlog(f"  Durbin-Watson (ens)  : {dw_ens:.4f}")

wlog(f"\n  R2 by horizon (ensemble):")
for h in [0, 4, 9, 21, 43, 65]:
    wlog(f"    h={h+1:>2d}: {r2_ens_by_h[h]:.4f}")

pd.DataFrame([{
    "Model":              "HAR-X + LSTM Ensemble",
    "R2_h1":              r2_ens_h1,  "R2_h22": r2_ens_h22,  "R2_h66": r2_ens_h66,
    "RMSE_log_h1":        rmse_ens,   "RMSE_pct_h1": rmse_pct_e,
    "MAE_log_h1":         mae_ens,    "MAPE_h1": mape_ens,  "DA_h1": da_ens,
    "DW_stat":            dw_ens,
    "HAR_R2_h1":          r2_har_h1,  "HAR_RMSE_log_h1": rmse_har,
    "LSTM_best_epoch":    best_epoch, "LSTM_best_val_loss": best_val,
    "R2_improvement_h1":  r2_improvement,
}]).to_csv("evaluation_metrics.csv", index=False)


# ==============================================================================
# SECTION 21: 3-MONTH FORECAST (SINGLE FORWARD PASS, NO RECURSION)
# ==============================================================================
# One call to har.predict → 66 HAR values (term structure)
# One call to resid_model.predict → 66 LSTM corrections
# Ensemble = sum of both in log space → exponentiate to vol%
# CI from empirical quantiles of HAR-X test residuals per horizon (log space)

last_x_sc     = scaler_tab.transform(feat[feature_cols].values[-1:])
fc_har_log    = har.predict(last_x_sc)[0]

last_resid    = residuals[-SEQ_LEN_RES:]
last_resid_sc = scaler_r.transform(last_resid)
fc_resid_sc   = resid_model.predict(
    last_resid_sc.reshape(1, SEQ_LEN_RES, HORIZON), verbose=0)[0]
fc_resid      = scaler_r.inverse_transform(fc_resid_sc.reshape(1, -1))[0]

fc_log     = fc_har_log + fc_resid
fc_vol     = np.exp(fc_log) * 100
fc_har_vol = np.exp(fc_har_log) * 100

har_test_resid = y_te - y_har_te
q025 = np.percentile(har_test_resid, 2.5,  axis=0)
q975 = np.percentile(har_test_resid, 97.5, axis=0)

upper_ci = np.exp(fc_log + q975) * 100
lower_ci = np.maximum(np.exp(fc_log + q025) * 100, 0.5)

bdays = pd.bdate_range(
    start=feat.index[-1] + pd.Timedelta(days=1),
    periods=HORIZON
)

wlog_sep("21. 3-MONTH FORECAST (HAR-X + LSTM ENSEMBLE)")
wlog(f"{'Day':>4}  {'Date':>12}  {'Ensemble':>10}  {'HAR-only':>10}  {'Lower':>8}  {'Upper':>8}")
wlog("-" * 64)
for i in [0, 4, 9, 21, 43, 65]:
    wlog(f"  {i+1:2d}  {str(bdays[i].date()):>12}  "
         f"{fc_vol[i]:>8.2f}%  {fc_har_vol[i]:>8.2f}%  "
         f"{lower_ci[i]:>6.2f}%  {upper_ci[i]:>6.2f}%")

wlog(f"\nForecast summary:")
wlog(f"  Range : {fc_vol.min():.2f}% to {fc_vol.max():.2f}%")
wlog(f"  Mean  : {fc_vol.mean():.2f}%")
wlog(f"  CI upper range: {upper_ci.min():.2f}% to {upper_ci.max():.2f}%")
wlog(f"  CI lower range: {lower_ci.min():.2f}% to {lower_ci.max():.2f}%")
wlog(f"  Current vol vs hist mean: {current_vol:.2f}% vs {hist_mean:.2f}%")
wlog(f"  Forecast direction: {'mean-reverting UP' if fc_vol.mean() > current_vol else 'mean-reverting DOWN'}")

pd.DataFrame({
    "Date": bdays, "Forecast_Vol": fc_vol, "HAR_X_Vol": fc_har_vol,
    "Upper_CI_95": upper_ci, "Lower_CI_95": lower_ci
}).to_csv("forecast_3month.csv", index=False)


# ==============================================================================
# SECTION 22: ALL CHARTS (saved to plots/ folder)
# ==============================================================================

def ppath(filename):
    return os.path.join(PLOTS_DIR, filename)

# -- Plot 01: FTSE Price + 5-day Vol (two-panel) --
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(df.index, df[TARGET_COL], color="royalblue", linewidth=1)
axes[0].set_ylabel("Price (GBP)", fontsize=11)
axes[0].set_title(f"{TARGET_NAME} Price and 5-Day Realized Volatility",
                  fontsize=13, fontweight="bold")
rv5_plot = np.exp(feat["ftse_log_rv5"]) * 100
axes[1].plot(feat.index, rv5_plot, color="darkorange", linewidth=1)
axes[1].axhline(hist_mean, linestyle="--", color="navy",
                linewidth=1.2, alpha=0.7, label=f"Mean ({hist_mean:.1f}%)")
axes[1].set_ylabel("5-Day Realized Vol (%)", fontsize=11)
axes[1].set_xlabel("Date", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(ppath("01_ftse_price_vol.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 02: Global Index Normalized Price Comparison --
# Normalize each index to 100 at the common start date.
# Shows relative performance and co-movement across the full sample.
df_common = df[AVAIL_COLS].dropna()
df_norm   = df_common / df_common.iloc[0] * 100

fig, ax = plt.subplots(figsize=(18, 8))
for i, col in enumerate(AVAIL_COLS):
    ax.plot(df_norm.index, df_norm[col],
            label=INDICES[col], linewidth=1.2, color=f"C{i}", alpha=0.85)
ax.set_title("Global Equity Index Performance — Normalized to 100 at Start",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Normalized Price (base=100)", fontsize=11)
ax.legend(fontsize=9, ncol=2, loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(ppath("02_global_index_normalized.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 03: Global 5-Day Realized Volatility Comparison --
# Plots annualized 5-day realized vol for all indices on same axis.
# Shows vol spillover timing: peaks usually occur near-simultaneously.
rv5_df_plot = pd.DataFrame(
    {INDICES[c]: np.exp(np.log(np.log(df[c]/df[c].shift(1)).rolling(5).std()*np.sqrt(252)+1e-8))*100
     for c in AVAIL_COLS}, index=df.index).dropna()

# Simpler: compute rv5 directly
rv5_plot_df = pd.DataFrame(index=df.index)
for c in AVAIL_COLS:
    lr_c = np.log(df[c] / df[c].shift(1))
    rv5_plot_df[INDICES[c]] = lr_c.rolling(5).std() * np.sqrt(252) * 100
rv5_plot_df = rv5_plot_df.dropna()

fig, ax = plt.subplots(figsize=(18, 7))
for i, col in enumerate(AVAIL_COLS):
    ax.plot(rv5_plot_df.index, rv5_plot_df[INDICES[col]],
            label=INDICES[col], linewidth=0.9, color=f"C{i}", alpha=0.8)
ax.set_title("5-Day Realized Volatility — 10 Global Indices (Annualized %)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Annualized Vol (%)", fontsize=11)
ax.legend(fontsize=9, ncol=2, loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(ppath("03_global_vol_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 04: Annual Mean Volatility Heatmap --
# Average 5-day RV per year per index, shows regime shifts across years.
rv5_plot_df["Year"] = rv5_plot_df.index.year
annual_vol = rv5_plot_df.groupby("Year")[[INDICES[c] for c in AVAIL_COLS]].mean()
fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(annual_vol.T, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=ax, linewidths=0.3, annot_kws={"size": 8})
ax.set_title("Annual Mean 5-Day Realized Volatility (%) by Index and Year",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(ppath("04_annual_vol_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 05: Vol Correlation Heatmap --
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.3, annot_kws={"size": 9})
ax.set_title("Volatility Correlation Matrix — 10 Global Indices (5-day RV)",
             fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(ppath("05_vol_corr_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 06: Rolling 90-day FTSE vs other indices vol correlation --
fig, ax = plt.subplots(figsize=(16, 6))
for i, col in enumerate(AVAIL_COLS):
    if col == TARGET_COL:
        continue
    rc = feat["ftse_log_rv5"].rolling(90).corr(
        pd.Series(rv5_plot_df[INDICES[col]].values,
                  index=rv5_plot_df.index).reindex(feat.index))
    ax.plot(feat.index, rc, linewidth=1, label=INDICES[col],
            color=f"C{i}", alpha=0.85)
ax.axhline(0, linestyle="--", color="black", linewidth=0.8)
ax.set_title(f"{TARGET_NAME} Rolling 90-Day Vol Correlation vs Other Indices",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Correlation")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(ppath("06_rolling_corr_ftse.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 07: FTSE Vol Percentile Rank --
pct_rank = feat["ftse_log_rv5"].rolling(252).rank(pct=True) * 100
fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(feat.index, pct_rank, alpha=0.3, color="steelblue")
ax.plot(feat.index, pct_rank, linewidth=0.8, color="steelblue")
ax.axhline(80, linestyle="--", color="red",   linewidth=1.2, label="80th pct (high vol)")
ax.axhline(20, linestyle="--", color="green", linewidth=1.2, label="20th pct (low vol)")
ax.set_title(f"{TARGET_NAME} Volatility Percentile Rank (252-Day Rolling)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Percentile (%)")
ax.set_ylim(0, 105); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("07_ftse_vol_percentile.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 08: ACF and PACF of FTSE log-vol --
acf_v  = acf(feat["ftse_log_rv5"].dropna(),  nlags=66, fft=True)
pacf_v = pacf(feat["ftse_log_rv5"].dropna(), nlags=40)
ci_bound = 1.96 / np.sqrt(n_total)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(len(acf_v)),  acf_v,  color="steelblue", alpha=0.75)
axes[1].bar(range(len(pacf_v)), pacf_v, color="darkorange", alpha=0.75)
for ax_i in axes:
    ax_i.axhline( ci_bound, linestyle="--", color="red", linewidth=1)
    ax_i.axhline(-ci_bound, linestyle="--", color="red", linewidth=1)
    ax_i.axhline(0, color="black", linewidth=0.5)
axes[0].set_title("ACF — FTSE log(rv5)  [slow decay = long memory]",
                  fontsize=11, fontweight="bold")
axes[1].set_title("PACF — FTSE log(rv5)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Lag"); axes[1].set_xlabel("Lag")
plt.tight_layout()
plt.savefig(ppath("08_acf_pacf.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 09: Feature correlation with target --
feat_corr = feat[feature_cols].corrwith(feat["ftse_log_rv5"]).sort_values()
colors    = ["red" if v < 0 else "steelblue" for v in feat_corr.values]
fig, ax = plt.subplots(figsize=(10, 6))
feat_corr.plot(kind="barh", ax=ax, color=colors, alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlation with FTSE log(rv5) Target",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Pearson Correlation")
plt.tight_layout()
plt.savefig(ppath("09_feature_target_corr.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 10: HAR-X Feature Importance (|Ridge coef| at h=1) --
fig, ax = plt.subplots(figsize=(10, 6))
coef_abs.sort_values().plot(kind="barh", ax=ax, color="steelblue", alpha=0.8)
ax.set_title("HAR-X Feature Importance — |Ridge Coefficient| at h=1",
             fontsize=13, fontweight="bold")
ax.set_xlabel("|Coefficient|")
plt.tight_layout()
plt.savefig(ppath("10_feature_importance.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 11: HAR-X R2 by forecast horizon --
r2_by_h = [r2_score(y_te[:, h], y_har_te[:, h]) for h in range(HORIZON)]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, HORIZON+1), r2_by_h, color="steelblue", linewidth=2)
ax.fill_between(range(1, HORIZON+1), 0, r2_by_h, alpha=0.2, color="steelblue")
ax.axhline(0,  linestyle="--", color="black", linewidth=0.8)
ax.axvline(22, linestyle="--", color="red",   linewidth=1.2, label="h=22 (1-month)")
ax.axvline(66, linestyle="--", color="green", linewidth=1.2, label="h=66 (3-month)")
ax.set_title("HAR-X R2 Score by Forecast Horizon (Test Set)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Forecast Horizon (trading days)"); ax.set_ylabel("R2")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("11_har_r2_by_horizon.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 12: Residual LSTM training loss --
fig, ax = plt.subplots(figsize=(12, 5))
ep_range = range(1, len(train_losses) + 1)
ax.plot(ep_range, train_losses, color="steelblue",  linewidth=1.5, label="Train Loss")
ax.plot(ep_range, val_losses,   color="darkorange", linewidth=1.5, label="Val Loss")
ax.axvline(best_epoch, linestyle="--", color="gray", linewidth=1.2,
           label=f"Best Weights Restored (epoch {best_epoch})")
ax.set_title(f"Residual LSTM Training — {EPOCHS} Epochs (No EarlyStopping)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Huber Loss")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("12_lstm_training_loss.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 13: Actual vs Predicted h=1 on test set --
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(har_test_dates, np.exp(y_te[:, 0])*100,
        label="Actual",   color="black",      linewidth=2.0)
ax.plot(har_test_dates, np.exp(y_har_te[:, 0])*100,
        label="HAR-X",    color="steelblue",  linewidth=1.6, linestyle="--", alpha=0.85)
ax.plot(ens_test_dates, np.exp(y_ens_te[:, 0])*100,
        label="Ensemble", color="darkorange", linewidth=1.8)
ax.set_title(f"Actual vs Predicted {TARGET_NAME} Vol (Test Set, h=1)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Ann. Vol (%)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(ppath("13_actual_vs_predicted.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 14: Scatter actual vs predicted (h=1) --
actual_v = np.exp(y_true_aln[:, 0]) * 100
pred_v   = np.exp(y_ens_te[:, 0])   * 100
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(actual_v, pred_v, alpha=0.35, s=14, color="steelblue")
mn = min(actual_v.min(), pred_v.min())
mx = max(actual_v.max(), pred_v.max())
ax.plot([mn, mx], [mn, mx], "r--", linewidth=1.5, label="Perfect fit")
ax.set_title(f"Predicted vs Actual Vol (Ensemble, h=1) | R2={r2_ens_h1:.4f}",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Actual Vol (%)"); ax.set_ylabel("Predicted Vol (%)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("14_scatter_predicted_actual.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 15: HAR residual distribution at h=1 --
resid_h1    = y_te[:, 0] - y_har_te[:, 0]
resid_jb_p  = jarque_bera(resid_h1)[1]
x_range     = np.linspace(resid_h1.min(), resid_h1.max(), 200)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(resid_h1, bins=50, color="steelblue", alpha=0.75,
        edgecolor="none", density=True, label="Residuals")
ax.plot(x_range,
        scipy_norm.pdf(x_range, resid_h1.mean(), resid_h1.std()),
        color="red", linewidth=1.8, label="Normal fit")
ax.axvline(0, linestyle="--", color="black", linewidth=1)
ax.set_title("HAR-X Residual Distribution (h=1) — Normality Check",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Residual (log-vol)"); ax.set_ylabel("Density")
ax.text(0.97, 0.93, f"JB p-val: {resid_jb_p:.4f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("15_residual_distribution.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 16: Main 3-month forecast --
hist_log = feat["ftse_log_rv5"].iloc[-120:]
hist_vol = np.exp(hist_log.values) * 100
y_max    = max(upper_ci.max(), hist_vol.max()) * 1.1

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(hist_log.index, hist_vol, color="black", linewidth=2,
        label="Historical Volatility (last 120 days)")
ax.plot([hist_log.index[-1]] + list(bdays),
        [hist_vol[-1]] + list(fc_har_vol),
        color="steelblue", linewidth=1.8, linestyle=":",
        label="HAR-X Component", alpha=0.85)
ax.plot([hist_log.index[-1]] + list(bdays),
        [hist_vol[-1]] + list(fc_vol),
        color="red", linewidth=2.5, linestyle="--",
        label="Ensemble Forecast (HAR-X + LSTM)")
ax.fill_between(bdays, lower_ci, upper_ci, alpha=0.15, color="red",
                label="95% CI (empirical quantiles, log-space)")
ax.axvline(x=feat.index[-1], linestyle=":", color="gray",
           linewidth=1.5, label="Forecast Start")
ax.axhline(y=hist_mean, linestyle="--", color="navy", linewidth=1.2, alpha=0.6,
           label=f"Historical Mean ({hist_mean:.1f}%)")
ax.set_ylim(0, y_max)
ax.set_title(
    f"{TARGET_NAME} Volatility — 3-Month Forecast\n"
    f"HAR-X Direct Multi-Step + LSTM Residual Correction | Source: Yahoo Finance",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Date", fontsize=12); ax.set_ylabel("Annualized Vol (%)", fontsize=12)
ax.legend(fontsize=10, loc="upper left"); ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(ppath("16_forecast_3month.png"), dpi=150, bbox_inches="tight")
plt.close()

# -- Plot 17: Weekly aggregated forecast bar chart --
fc_df         = pd.DataFrame({"Date": bdays, "Vol": fc_vol,
                               "Upper": upper_ci, "Lower": lower_ci})
fc_df["Week"] = pd.to_datetime(fc_df["Date"]).dt.to_period("W")
weekly        = fc_df.groupby("Week").mean(numeric_only=True).reset_index()
x_pos         = np.arange(len(weekly))
yerr_up       = np.maximum(weekly["Upper"] - weekly["Vol"], 0).values
yerr_lo       = np.maximum(weekly["Vol"]   - weekly["Lower"], 0).values
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_pos, weekly["Vol"], color="royalblue", alpha=0.7, label="Forecast Vol")
ax.errorbar(x_pos, weekly["Vol"], yerr=[yerr_lo, yerr_up],
            fmt="none", color="black", capsize=4, linewidth=1.2, label="95% CI")
ax.set_xticks(x_pos)
ax.set_xticklabels([str(w) for w in weekly["Week"]], rotation=45, fontsize=9)
ax.set_title(f"{TARGET_NAME} Weekly Mean Forecast Volatility (Next 3 Months)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Week"); ax.set_ylabel("Ann. Vol (%)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(ppath("17_weekly_forecast_bar.png"), dpi=150, bbox_inches="tight")
plt.close()

log.close()

C:\Users\dgoya\AppData\Local\Temp\ipykernel_3496\723218405.py:272: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_s, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
C:\Users\dgoya\AppData\Local\Temp\ipykernel_3496\723218405.py:272: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_s, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
C:\Users\dgoya\AppData\Local\Temp\ipykernel_3496\723218405.py:272: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_s, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
C:\Users\dgoya\AppData\Local\Temp\ipykernel_3496\723218405.py:272: InterpolationWarning: The test st